# Case study 2: One-Tap Apply v2, a cluster-randomized test

*Sequel to Case 1. Synthetic data, modeled on how two-sided job marketplaces run experiments.*

## The brief

Your Case 1 memo said: don't ship One-Tap Apply as-is, keep required screening questions, and **randomize by job posting next time** so recruiter-side effects are measured cleanly. Leadership agreed.

**One-Tap v2:** members still apply in one tap, but required screening questions stay. Only optional questions are skipped.

**Design:** every job posting created between Nov 30 and Dec 6, 2026 was randomly assigned to treatment or control **when it was created**. Everyone who applies to a treatment posting sees One-Tap v2. Applications were tracked for 4 weeks.

The PM writes:

> "v2 is live on half of postings. Quick heads-up: the dashboard says recruiter response rate is significantly down again (p = 0.002). Should we kill it?"

**Your job:** figure out whether that's real, measure the effect on applications and quality correctly, and make a launch call.

## What's new in this case

The unit of randomization is now a **cluster** (a job posting) that contains many applications. Applications to the same posting share a recruiter, a company and a job, so their outcomes are correlated. That changes how you have to analyze every metric.

## Data (in `data/`)

| File | Grain | Columns |
|---|---|---|
| `postings.csv` | 1 row per job posting in the test | `posting_id`, `variant`, `created_date`, `job_function`, `seniority`, `company_size`, `company_avg_apps_pre` (avg applications per posting this company got in the 90 days **before** the test), `recruiter_resp_rate_pre` (this recruiter's response rate in the 90 days **before** the test) |
| `applications.csv` | 1 row per application | `application_id`, `posting_id`, `member_id`, `applied_date`, `recruiter_responded_7d`, `recruiter_marked_qualified` (recruiter tagged the applicant as a fit) |

## Methods you'll practice

| New | Review from Case 1 |
|---|---|
| Intraclass correlation (ICC) | SRM check |
| Design effect and effective sample size | Welch t-test |
| Cluster-level analysis | CUPED |
| Cluster bootstrap | Delta method for ratio metrics |
| Covariate balance (standardized mean difference) | Launch memo |
| Power for a cluster-randomized test | |

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
DATA = "data/"
ALPHA = 0.05

---
# Part A: Design questions (answer before touching data)

**A1.** Randomizing by posting fixes the recruiter-side interference from Case 1. What new problem does it create on the **member** side? Which direction would it bias the applications result?

> Randomizing by posting fixes recruiter-side interference, but it risks a SUTVA violation on the member side. SUTVA assumes each unit's outcome depends only on its own assignment, so control postings should behave as if One-Tap didn't exist. Here, members aren't randomized: the same member sees both One-Tap and regular postings. A member who would have applied to a control posting may apply to an easier One-Tap posting instead. That shifts applications from control to treatment, inflating one and deflating the other, so the measured lift likely overstates the true effect. After a full launch, every posting has One-Tap, so there's nothing easier to shift to, and only genuinely new applications count.

**A2.** For each metric below, what is the unit of analysis, and is it a simple mean or a ratio metric?
- Applications per posting (primary)
> Unit of Analysis: Posting || Simple Mean ✅

- Recruiter response rate (guardrail)
> Unit of Analysis: Applications || Ratio Metric ✅


- Qualified applicants per posting (employer-side "north star")

> Unit of Analysis: Posting || Simple Mean ✅

**A3.** The test has about 4,000 postings and over 100,000 applications. Which number is your sample size, and why?

> 4,000 postings will be the sample size because that was the units that were used for randomization.

**A4.** What would you use the two pre-period columns for?

> The reason why we have two pre-period columns is because due to a small sample of 4,000 postings - there is a likelihood that there could be an imbalance between the arms (Control v Treatment). The reason that we have the two pre-period columns `company_avg_apps_pre` and `recruiter_resp_rate_pre` is because the previous can predict the in-experiement metric during the test. CUPED, will reduce the variance without having to increase sample size or increase the duration of the test.

**Your answer:**

_Write it here before looking at the answer key._

---
# Part B: Trust checks

## B1. Build a posting-level table

One row per posting from `postings.csv`, with:
- `apps`: number of applications
- `responses`: sum of `recruiter_responded_7d`
- `qualified`: sum of `recruiter_marked_qualified`

Postings with no applications must stay in, with zeros.

In [53]:
postings = pd.read_csv(DATA + "postings.csv")
apps = pd.read_csv(DATA + "applications.csv")

apps.head() 

,posting_id,apps,responses,qualified
0,3000154123,128,18,29
1,3000306118,13,1,1
2,3000631024,35,0,11
3,3001679734,69,21,14
4,3002259564,87,3,14


In [54]:
postings.head()

,posting_id,variant,created_date,job_function,seniority,company_size,company_avg_apps_pre,recruiter_resp_rate_pre
0,3158952705,treatment,2026-12-02,engineering,mid,mid,20.0000,0.1250
1,3694666111,treatment,2026-12-01,operations,senior,mid,7.9000,0.1110
2,3228567779,treatment,2026-12-04,sales,mid,small,21.7000,0.3180
3,3890683883,treatment,2026-12-05,sales,mid,small,19.9000,0.4420
4,3396841127,treatment,2026-12-03,marketing,entry,mid,37.7000,0.0500


In [61]:
postings = pd.read_csv(DATA + "postings.csv")
apps = pd.read_csv(DATA + "applications.csv")

postings.head() 

# TODO: aggregate apps by posting_id, left-join onto postings, fill zeros
apps_agg = apps.groupby('posting_id').agg(
    apps = ('posting_id', 'size'), 
    responses = ('recruiter_responded_7d', 'sum'), 
    qualified = ('recruiter_marked_qualified', 'sum')
).reset_index() 

posts = postings.merge(apps_agg, on = 'posting_id', how = 'left')
cols = ['apps', 'responses', 'qualified']

posts[cols] = posts[cols].fillna(0).astype(int) 

posts.head() 

,posting_id,variant,created_date,job_function,seniority,company_size,company_avg_apps_pre,recruiter_resp_rate_pre,apps,responses,qualified
0,3158952705,treatment,2026-12-02,engineering,mid,mid,20.0000,0.1250,23,1,13
1,3694666111,treatment,2026-12-01,operations,senior,mid,7.9000,0.1110,4,0,1
2,3228567779,treatment,2026-12-04,sales,mid,small,21.7000,0.3180,21,4,3
3,3890683883,treatment,2026-12-05,sales,mid,small,19.9000,0.4420,41,16,9
4,3396841127,treatment,2026-12-03,marketing,entry,mid,37.7000,0.0500,30,0,8


## B2. SRM on postings

Same chi-square check as Case 1, but count **postings** (the randomization unit), not applications.

**Question:** why would an SRM check on application counts be misleading here?

In [51]:
# TODO: chi-square goodness of fit on posting counts by variant
def srm_test(counts): 
    expected = np.full(len(counts), sum(counts)/len(counts))
    return stats.chisquare(counts.values, expected).pvalue

posts_counts = posts['variant'].value_counts()

# Test once - avoid multiple testing
p = srm_test(posts_counts)

print(posts_counts.to_dict(), f"Treatment Share: {posts_counts['treatment']/posts_counts.sum():.4}", f"SRM p-value: {p:.4f}")  

if p > 0.05: 
    print("No SRM detected")

{'control': 2015, 'treatment': 1985} Treatment Share: 0.4963 SRM p-value: 0.6353
Balanced


**Checkpoint:** p should be well above 0.05.

## B3. Covariate balance

With fewer randomized units, chance imbalances are bigger. Check that the arms look alike **before** treatment using the **standardized mean difference (SMD)**:

SMD = (mean_T − mean_C) / √((var_T + var_C) / 2)

A common rule of thumb: |SMD| < 0.1 is balanced. Check the two pre-period numbers, plus the share of postings in each `company_size` and `seniority` category (use 0/1 dummies).

In [55]:
postings.head()

,posting_id,variant,created_date,job_function,seniority,company_size,company_avg_apps_pre,recruiter_resp_rate_pre
0,3158952705,treatment,2026-12-02,engineering,mid,mid,20.0000,0.1250
1,3694666111,treatment,2026-12-01,operations,senior,mid,7.9000,0.1110
2,3228567779,treatment,2026-12-04,sales,mid,small,21.7000,0.3180
3,3890683883,treatment,2026-12-05,sales,mid,small,19.9000,0.4420
4,3396841127,treatment,2026-12-03,marketing,entry,mid,37.7000,0.0500


In [62]:
def smd(df, col):
    t = df.loc[df["variant"] == "treatment", col]
    c = df.loc[df["variant"] == "control", col]
  
    return (t.mean() - c.mean()) / np.sqrt((t.var() + c.var()) / 2)

bal = postings[["variant", "company_avg_apps_pre", "recruiter_resp_rate_pre"]].join(
    pd.get_dummies(posts[["company_size", "seniority"]], dtype = int)
)

balance_test_results = pd.Series({col : smd(bal, col) for col in bal.columns if col != "variant"}, name = "SMD_Check")

print(balance_test_results)

# TODO: SMD for company_avg_apps_pre, recruiter_resp_rate_pre, and dummies of company_size and seniority
company_avg_apps_pre = smd(postings, "company_avg_apps_pre") 


company_avg_apps_pre      -0.0002
recruiter_resp_rate_pre   -0.0151
company_size_large        -0.0192
company_size_mid           0.0165
company_size_small        -0.0007
seniority_entry           -0.0431
seniority_mid              0.0721
seniority_senior          -0.0386
Name: SMD_Check, dtype: float64


**Checkpoint:** every |SMD| should be below 0.1.

Negative values mean that the mean of the treatment group is higher than the control. The only thing that is close to the margin is `seniority_mid` and that is 0.07, but there are slightly fewer in the treatment for the `seniority_entry` and the `seniority_senior` group. 

---
# Part C: The tempting (wrong) analysis

This is roughly what the PM's dashboard did. Treat every application as an independent observation and run a two-proportion z-test on:
1. Recruiter response rate
2. Qualified rate (qualified / applications)

In [63]:
apps.head()

,application_id,posting_id,member_id,applied_date,recruiter_responded_7d,recruiter_marked_qualified
0,700029158,3571899262,88414188,2026-12-05,0,0
1,700071351,3083243038,85605813,2026-12-10,1,0
2,700008983,3575786061,88414188,2026-12-13,0,0
3,700026142,3051075839,39788440,2026-12-16,1,0
4,700067504,3117231334,76520886,2026-12-08,0,0


In [64]:
postings.head() 

,posting_id,variant,created_date,job_function,seniority,company_size,company_avg_apps_pre,recruiter_resp_rate_pre
0,3158952705,treatment,2026-12-02,engineering,mid,mid,20.0000,0.1250
1,3694666111,treatment,2026-12-01,operations,senior,mid,7.9000,0.1110
2,3228567779,treatment,2026-12-04,sales,mid,small,21.7000,0.3180
3,3890683883,treatment,2026-12-05,sales,mid,small,19.9000,0.4420
4,3396841127,treatment,2026-12-03,marketing,entry,mid,37.7000,0.0500


In [84]:
apps_p = apps.merge(postings[['posting_id', 'variant']], on='posting_id', how='left')

### Formulas: pooled two-proportion z-test

**Notation** (per arm: C = control, T = treatment)
- $x$ = number of successes (applications with the outcome = 1) → `sum`
- $n$ = number of applications → `count`

**1. Rate in each arm**
$$\hat{p}_C = \frac{x_C}{n_C}, \qquad \hat{p}_T = \frac{x_T}{n_T}$$

**2. Pooled rate** (best estimate of the single shared rate if $H_0$: no difference is true)
$$\hat{p}_{\text{pool}} = \frac{x_C + x_T}{n_C + n_T}$$

**3. Standard error of the difference**
$$\text{SE} = \sqrt{\hat{p}_{\text{pool}}\,(1 - \hat{p}_{\text{pool}})\left(\frac{1}{n_C} + \frac{1}{n_T}\right)}$$

**4. z-score** (how many SEs the difference is from 0; convention is treatment − control)
$$z = \frac{\hat{p}_T - \hat{p}_C}{\text{SE}}$$

**5. Two-sided p-value**
$$p = 2\,\big(1 - \Phi(|z|)\big) \quad \longrightarrow \quad \texttt{2 * stats.norm.sf(abs(z))}$$

**Why it's "naive" here:** $n_C$ and $n_T$ count *applications* (~141k total) as if each were an independent draw. Applications to the same posting share a recruiter, so they're correlated, which makes this SE too small and the p-value too optimistic. Part D measures how much.


In [119]:
def naive_prop_test(df_apps, col):
    g = df_apps.groupby("variant")[col].agg(["sum", "count"])

    x_c, n_c = g.loc["control", "sum"],   g.loc["control", "count"]
    x_t, n_t = g.loc["treatment", "sum"], g.loc["treatment", "count"]

    p_c, p_t = x_c / n_c, x_t / n_t
    p_pool = (x_c + x_t) / (n_c + n_t)

    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
    z = (p_t - p_c) / se
    p_value = 2 * stats.norm.sf(abs(z))

    return {"control": p_c, "treatment": p_t, "diff": p_t - p_c,
            "se": se, "z": z, "p_value": p_value}

apps_v = apps.merge(postings[["posting_id", "variant"]], on="posting_id")
naive = pd.DataFrame({
    "response rate": naive_prop_test(apps_v, "recruiter_responded_7d"),
    "qualified rate": naive_prop_test(apps_v, "recruiter_marked_qualified"),
}).T
naive


,control,treatment,diff,se,z,p_value
response rate,0.1799,0.1737,-0.0063,0.0020,-3.0813,0.0021
qualified rate,0.2884,0.2942,0.0058,0.0024,2.3875,0.0170


**Checkpoint:** the response rate should look significantly down (p ≈ 0.002), just like the PM said.

**Question:** before reading on, write down what assumption this test makes that's violated here.

Baesd on the `response_rate` metric - this should be when the applications are randomized. In this case, the job postings are randomized. Thus, would be a bad metric to use. The z-test will have to make sure that the applications are from an independent draw.

---
# Part D: Measuring the clustering (ICC and design effect)

## D1. Intraclass correlation (ICC)

ICC is the share of the total variation in an outcome that comes from **differences between clusters**. ICC = 0 means postings don't matter (applications behave like independent coin flips). ICC = 1 means every application to the same posting gets the same outcome.

Estimate it with one-way ANOVA across postings (use all applications, both arms):

- MSB = between-posting mean square = Σ nᵢ (ȳᵢ − ȳ)² / (k − 1)
- MSW = within-posting mean square = Σ (y − ȳᵢ)² / (N − k)
- m₀ = (N − Σ nᵢ² / N) / (k − 1), an "average" cluster size adjusted for unequal sizes
- **ICC = (MSB − MSW) / (MSB + (m₀ − 1) · MSW)**

where k = number of postings with at least 1 application, nᵢ = applications to posting i, and N = total applications.

In [88]:
apps_v = apps.merge(postings[['posting_id', 'variant']], on = 'posting_id', how = 'left')

apps_v.head() 

,application_id,posting_id,member_id,applied_date,recruiter_responded_7d,recruiter_marked_qualified,variant
0,700029158,3571899262,88414188,2026-12-05,0,0,treatment
1,700071351,3083243038,85605813,2026-12-10,1,0,treatment
2,700008983,3575786061,88414188,2026-12-13,0,0,control
3,700026142,3051075839,39788440,2026-12-16,1,0,control
4,700067504,3117231334,76520886,2026-12-08,0,0,control


In [103]:
def icc_anova(df, cluster, y):
    g = df.groupby(cluster)[y].agg(['mean', 'count'])

    k, N = len(g), g['count'].sum()
    y_bar_grand = df[y].mean()

    msb = (g['count'] * (g['mean'] - y_bar_grand) ** 2).sum() / (k - 1)

    own_mean = df.groupby(cluster)[y].transform('mean')
    msw = ((df[y] - own_mean) ** 2).sum() / (N - k)

    m0 = (N - (g['count'] ** 2).sum() / N) / (k - 1)

    return (msb - msw) / (msb + (m0 - 1) * msw)


icc_resp = icc_anova(apps_p, "posting_id", "recruiter_responded_7d")
icc_qual = icc_anova(apps_p, "posting_id", "recruiter_marked_qualified")
print(f"ICC response = {icc_resp:.3f}, ICC qualified = {icc_qual:.3f}")


ICC response = 0.093, ICC qualified = 0.084


We randomized at the posting level, but recruiter_responded_7d is measured per application. Applications to the same posting share a recruiter, so their outcomes are likely correlated with each other. The ICC measures how strong that within-posting correlation is. It matters because the naive z-test in Part C treated every application as independent. If the ICC is above 0, the ~140k applications contain less independent information than they appear to, so the naive SE is too small and the p-value is too optimistic.

## D2. Design effect

The **design effect (DEFF)** is how much clustering inflates the variance compared to independent observations. With unequal cluster sizes, a good approximation is:

DEFF = 1 + (m_w − 1) · ICC, where m_w = Σ nᵢ² / Σ nᵢ (the size-weighted average cluster size)

Then:
- **Effective sample size** = N / DEFF
- **Correct SE ≈ naive SE × √DEFF**

Compute DEFF for response rate and apply it to the naive result from Part C. Is the drop still significant?

In [127]:
# TODO: m_w from cluster sizes, DEFF, effective sample size

n = apps_p.groupby('posting_id').size() 

m_w = (n **2).sum() / n.sum() 

DEFF_resp = 1 + (m_w - 1) * icc_resp 

print(DEFF_resp)

print(f"Size Weight Average Cluster Size: {m_w:.4f}")
print(f"DEFF = {DEFF_resp:.4f} | Meaning that {len(apps):,.0f} applications acts as {len(apps)/DEFF_resp:,.4f} independent ones")

r = naive.loc['response rate']

se_adj = r['se'] * np.sqrt(DEFF_resp) 
z_adj = (r['treatment'] - r['control']) / se_adj

print(f"Adjusted SE = {se_adj:.4f} | Adjusted Z-Score : {z_adj:.4f} | Adjusted P-Value : {2 * stats.norm.sf(abs(z_adj)):.3f}")


# TODO: inflate the naive SE for response rate, recompute z and p

9.044877426521117
Size Weight Average Cluster Size: 87.9315
DEFF = 9.0449 | Meaning that 141,529 applications acts as 15,647.4204 independent ones
Adjusted SE = 0.0061 | Adjusted Z-Score : -1.0246 | Adjusted P-Value : 0.306


**Checkpoint:** ICC for response rate should be around 0.1, but DEFF should be much bigger (around 9), because some postings get hundreds of applications.

**Question:** the ICC looks small. Why is the design effect still so big?

**Answer** 
ICC ≈ 0.09 for response and 0.08 for qualified. That sounds small, but DEFF multiplies it by the cluster size, and the size-weighted cluster size is large (~88) because big postings hold a lot of the applications. DEFF ≈ 9, so 141,500 applications carry about as much information as ~15,600 independent ones, and the correct SE is about √9 = 3x the naive one. After adjusting, the response rate drop is no longer significant (p ≈ 0.31).

**The general lesson:** even a small ICC is a big deal when clusters are large.

---
# Part E: The right way, two methods that should agree

## E1. Delta method at the posting level

Response rate = Σ responses / Σ applications is a **ratio metric**, and the posting is the randomization unit. That's exactly the Case 1 delta method setup, with posting in place of member. Reuse your `ratio_delta` function on the posting-level table.

In [137]:
def ratio_delta(df, num, den):
    y, x = df[num] , df[den]
    r = y.sum() / x.sum() 

    var = (y.var() - 2 * r * np.cov(y, x)[0,1] + r**2 * x.var()) / (len(df)*x.mean()**2)

    return r, var 


def compare_ratio(df, num, den):
    r_c, v_c = ratio_delta(df[df["variant"] == "control"], num, den)
    r_t, v_t = ratio_delta(df[df["variant"] == "treatment"], num, den)
    se = np.sqrt(v_c + v_t)
    return {"control": r_c, "treatment": r_t, "rel_diff": r_t / r_c - 1, "se": se, "p_value": 2 * stats.norm.sf(abs((r_t - r_c) / se))}


delta = pd.DataFrame({
    "response rate": compare_ratio(posts, "responses", "apps"),
    "qualified rate": compare_ratio(posts, "qualified", "apps"),
}).T

delta

,control,treatment,rel_diff,se,p_value
response rate,0.1799,0.1737,-0.0347,0.0060,0.3011
qualified rate,0.2884,0.2942,0.0200,0.0071,0.4168


## E2. Cluster bootstrap

No formulas: resample **postings** (never individual applications) with replacement within each arm, recompute each arm's rate, and repeat. The spread of the differences is the SE.

Write `cluster_bootstrap(df, num, den, B=2000)` that returns the bootstrap SE and a 95% percentile interval for the difference in rates.

**Hint (fast version):** for each arm, draw an index matrix `rng.integers(0, n, size=(B, n))`, then the rate for each replicate is `y[idx].sum(axis=1) / x[idx].sum(axis=1)`.

### Notes 

Should get the same answer as E1 without a formula. --> Using bootstrapping --> what if i'd drawn a different set of postings? 

In [138]:
def cluster_bootstrap(df, num, den, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    rates = {}

    for v in ["control", "treatment"]:
        arm = df[df["variant"] == v]
        y, x = arm[num].to_numpy(), arm[den].to_numpy()       # ← added ()
        n = len(y)
        idx = rng.integers(0, n, size=(B, n))
        rates[v] = y[idx].sum(axis=1) / x[idx].sum(axis=1)   # one rate per resample

    diffs = rates["treatment"] - rates["control"]
    return diffs.std(), (np.percentile(diffs, 2.5), np.percentile(diffs, 97.5))   # ← (se, (low, high))


rows = {}
for name, num in [("response rate", "responses"), ("qualified rate", "qualified")]:
    se_b, ci = cluster_bootstrap(posts, num, "apps")
    rows[name] = {
        "naive SE": naive.loc[name, "se"],
        "delta SE": delta.loc[name, "se"],
        "bootstrap SE": se_b,
        "boot CI low": ci[0],
        "boot CI high": ci[1],
        "delta / naive": delta.loc[name, "se"] / naive.loc[name, "se"],
    }
pd.DataFrame(rows).T


,naive SE,delta SE,bootstrap SE,boot CI low,boot CI high,delta / naive
response rate,0.0020,0.0060,0.0061,-0.0179,0.0057,2.9796
qualified rate,0.0024,0.0071,0.0070,-0.0083,0.0193,2.9402


**Checkpoint:** delta and bootstrap SEs should be close to each other and about 3x the naive SE. The bootstrap CI for response rate should include 0.

**Question:** what's your verdict on the PM's "response rate is significantly down"?

The response rate has a p-value of 0.30 and the confidence interval includes zero. Thus, we would conclude that this is a false alarm due to the effect of treating clustered applications as independent. 

## E3. Which average? (a judgment question)

Σ responses / Σ applications weights every **application** equally, so a few postings with hundreds of applicants dominate it. The alternative is the **average of each posting's own response rate**, which weights every **posting (employer)** equally.

Compute the posting-average response rate for each arm (postings with at least 1 application), with a Welch t-test. Which version would you report to the Jobs leadership, and why?

In [149]:
# TODO: per-posting response rate, then compare means across arms


p1 = posts.loc[posts['apps'] > 0].copy() 

p1['response_rate'] = p1['responses'] / p1['apps']

t = p1.loc[p1['variant'] == 'treatment', 'response_rate']
c = p1.loc[p1['variant'] == 'control',  'response_rate']

print(f"Posting-average response rate: control {c.mean():.4f}, treatment {t.mean():.4f}, p = {stats.ttest_ind(t, c, equal_var=False).pvalue:.3f}")
print(f"Share of all applications in the top 5% largest postings: {p1['apps'].nlargest(int(len(p1) * 0.05)).sum() / p1['apps'].sum():.1%}")

Posting-average response rate: control 0.1805, treatment 0.1726, p = 0.118
Share of all applications in the top 5% largest postings: 24.9%


**Your answer:**

For E1, we see a control of 0.1799 and treatment of 0.1737 with a significance value of 0.30 and the posting-weighted option we see that the control is 0.1805 and the treatment is 0.1726 with a p-value of 0.118. Both are statistically insignificant. 

In E3 (Posting Weighted), may be a better metric to use as a primary metric because recruiter response rate is an employer-side guardrail. We want to makes sure that the employers are actually getting good applications with the one-tap apply. Thus, it would be useful to see if the one-tap apply is hurting recruiters.

---
# Part F: The primary metrics, done right

## F1. Applications per posting

Posting-level Welch t-test (the posting is both the randomization and analysis unit, so this is the easy case). Then add CUPED with `company_avg_apps_pre`. Reuse your Case 1 functions.

In [ ]:
def compare_means(df, col):
    # TODO (from Case 1)
    raise NotImplementedError


def cuped(df, y, x):
    # TODO (from Case 1)
    raise NotImplementedError

# TODO: raw and CUPED for apps; print rho and SE ratio

**Checkpoint:** raw lift ≈ +7% but not significant (p ≈ 0.10). CUPED should make it significant.

## F2. Qualified applicants per posting (the employer north star)

This is what employers actually want: more good applicants. Run raw and CUPED (same covariate).

In [ ]:
# TODO

**Checkpoint:** qualified applicants per posting should move up by a similar percentage to applications.

**Question:** how do F1, F2 and Part E fit together?

---
# Part G: Launch memo

About 150 words: decision, evidence, caveats, next steps. Include one sentence that explains the PM's false alarm in plain English.

**Your memo:**

_Decision:_

_Evidence:_

_Caveats:_

_Next steps:_

---
# Bonus: Interviewer follow-ups

1. **"Explain the design effect to a PM in one sentence."**
2. **"How would you measure the seeker-side cannibalization you mentioned?"**
3. **"For the next posting-level test, how many postings do you need to detect a 5% lift in applications per posting with 80% power?"** (Code it below: use the CV of `apps` per posting and the ρ from CUPED.)
4. **"Why not just randomize by recruiter or by company instead of by posting?"**

In [ ]:
# Stretch: n postings per arm = 2 (z_a + z_b)^2 * CV^2 * (1 - rho^2) / r^2
# TODO: compute with and without CUPED for r = 5%

_Your notes:_